# TP 5 — Isolation Forest sur des transactions réelles
## Séance 5 : Détection non supervisée

| | |
|---|---|
| **École** | ESAIP |
| **Classe** | Aix IR 5 CP |
| **Cours** | Détection d’anomalies et fraude avec Python |
| **Enseignant** | Serigne Faye |

## Contexte

Vous allez détecter des fraudes **sans utiliser aucune étiquette pour entraîner** le modèle, avec l’**Isolation Forest**, sur un jeu de données réel et public : **Credit Card Fraud Detection** (ULB Machine Learning Group et Worldline).

- 284 807 transactions de porteurs de cartes européens (deux jours de septembre 2013) ;
- 492 fraudes, soit 0,17 % des transactions ;
- 28 variables `V1` à `V28`, résultat d’une analyse en composantes principales (anonymisées), ainsi que `Time` (secondes écoulées depuis la première transaction), `Amount` (montant) et `Class` (1 = fraude, 0 = normale).

**Déroulé**

1. découvrir les données ;
2. préparer un découpage temporel (entraînement / test) ;
3. entraîner un Isolation Forest sans étiquettes ;
4. évaluer le classement des transactions (ROC-AUC, précision moyenne, rappel selon le nombre d’alertes) ;
5. choisir un seuil d’alerte ;
6. étudier l’effet des hyperparamètres et du choix des variables ;
7. inspecter les alertes.

## Étape 1 — Récupération du jeu de données

1. Téléchargez le fichier `creditcard.csv` sur Kaggle (jeu « Credit Card Fraud Detection », ULB Machine Learning Group). Il pèse environ 150 Mo.
2. Placez-le **dans le même dossier que ce notebook** (sur Google Colab : panneau « Fichiers », puis « Importer »).
3. Exécutez la cellule suivante **sans la modifier**.

Si le fichier est introuvable, la cellule crée un **jeu de secours simulé** de même structure (100 000 transactions) afin que le notebook reste exécutable. Les résultats ne seront alors pas ceux du jeu réel : un message d’avertissement s’affiche.

In [ ]:
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluer(y, alerte, afficher=True):
    alerte = np.asarray(alerte).astype(int)
    y = np.asarray(y)
    tp = int(((alerte == 1) & (y == 1)).sum())
    fp = int(((alerte == 1) & (y == 0)).sum())
    fn = int(((alerte == 0) & (y == 1)).sum())
    tn = int(((alerte == 0) & (y == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    rappel = tp / (tp + fn)
    if afficher:
        print(f'Alertes={tp + fp}  TP={tp}  FP={fp}  FN={fn}  TN={tn}')
        print(f'Précision={precision:.2f}  Rappel={rappel:.2f}')
    return precision, rappel

def precision_rappel_topk(y, score, k):
    # précision et rappel si l'on examine les k transactions au score le plus élevé
    y = np.asarray(y)
    meilleurs = np.argsort(-np.asarray(score))[:k]
    tp = int(y[meilleurs].sum())
    return tp / k, tp / int(y.sum())

def generer_secours(n=100_000, seed=0):
    # Jeu SIMULÉ de même structure que creditcard.csv (Time, V1..V28, Amount, Class)
    rng = np.random.default_rng(seed)
    sigma = np.linspace(1.9, 0.35, 28)
    V = rng.normal(0, sigma, size=(n, 28))
    y = (rng.random(n) < 0.0017).astype(int)
    for j, d in {2: -1.8, 3: 1.6, 6: -1.2, 9: -1.8, 10: 1.4, 11: -1.8, 13: -2.0, 15: -1.2, 16: -1.8, 17: -1.0}.items():
        V[y == 1, j] += d * sigma[j]
    montant = np.where(y == 1, rng.lognormal(3.5, 1.6, n), rng.lognormal(3.3, 1.4, n)).clip(0, 25000).round(2)
    temps = np.sort(rng.uniform(0, 172792, n)).round(0)
    out = pd.DataFrame(V, columns=[f'V{i}' for i in range(1, 29)])
    out.insert(0, 'Time', temps)
    out['Amount'] = montant
    out['Class'] = y
    return out

if os.path.exists('creditcard.csv'):
    df = pd.read_csv('creditcard.csv')
    print('Jeu de données RÉEL chargé :', df.shape)
else:
    df = generer_secours()
    print('ATTENTION : creditcard.csv est introuvable. Utilisation du jeu de SECOURS SIMULÉ :', df.shape)

## Étape 2 — Découverte des données

Complétez la cellule (remplacez chaque `...`).

In [ ]:
print(df.shape)
print(df['Class'].value_counts())

taux_fraude = ...
print(f'Taux de fraude : {taux_fraude:.3%}')

# Statistiques descriptives du montant
...

## Étape 3 — Préparation : variables et découpage temporel

- Les variables d’entrée sont `V1` à `V28` et `Amount`. La colonne `Class` n’est **jamais** une variable d’entrée.
- On ne standardise pas : l’Isolation Forest est insensible à l’échelle des variables.
- Découpage **dans le temps** : les 70 % de transactions les plus anciennes (selon `Time`) pour l’entraînement, les 30 % les plus récentes pour le test.

In [ ]:
variables = [c for c in df.columns if c.startswith('V')] + ['Amount']

seuil_temps = df['Time'].quantile(0.7)
train = ...      # transactions dont Time <= seuil_temps
test = ...       # transactions dont Time > seuil_temps

X_train = ...
X_test = ...
y_test = ...

print('Entraînement :', len(train), 'transactions |', 'Test :', len(test), 'transactions')
print('Fraudes dans le test :', int(y_test.sum()), f'({y_test.mean():.3%})')

## Étape 4 — Entraînement de l’Isolation Forest (sans étiquettes)

Rappel : `score_samples` renvoie l’**opposé** du score d’anomalie. On utilise `−score_samples` pour que « plus haut = plus anormal ».

In [ ]:
iso = IsolationForest(n_estimators=100, max_samples=256, contamination='auto', random_state=42, n_jobs=-1)
iso.fit(X_train)

score_test = ...                 # -score_samples sur X_test : plus haut = plus anormal
alerte_auto = ...                # True si predict(X_test) == -1

print('Alertes avec contamination=auto :', alerte_auto.sum(), 'sur', len(X_test), f'({alerte_auto.mean():.1%})')
evaluer(y_test, alerte_auto)

## Étape 5 — Évaluation du classement

On utilise maintenant les étiquettes du jeu de **test** (jamais vues par le modèle) pour juger la qualité du classement.

In [ ]:
auc = ...                        # roc_auc_score
ap = ...                         # average_precision_score
print(f'ROC-AUC = {auc:.3f}')
print(f'Précision moyenne = {ap:.3f}   (référence du hasard = taux de fraude du test = {y_test.mean():.4f})')

In [ ]:
# Précision et rappel si l'on examine les k transactions les plus suspectes
for k in [50, 100, 250, 500, 1000]:
    p, r = ...                   # precision_rappel_topk(y_test, score_test, k)
    print(f'k = {k:5d}  précision = {p:.2f}  rappel = {r:.2f}')

In [ ]:
# Fraudes retrouvées (%) selon la part des transactions examinées (les plus suspectes d'abord)
ordre = ...                      # indices triés par score décroissant
cumul = ...                      # nombre cumulé de fraudes retrouvées, divisé par le total de fraudes
part = np.arange(1, len(ordre) + 1) / len(ordre) * 100

plt.figure(figsize=(7, 3.8))
...
plt.xlim(0, 10)
plt.xlabel('Part des transactions examinées (%)')
plt.ylabel('Fraudes retrouvées (%)')
plt.show()

## Étape 6 — Choisir le seuil d’alerte

En production, on ne connaît pas les étiquettes des nouvelles transactions. On fixe donc un seuil sur les scores de l’**entraînement** (par exemple, le seuil qui signale 1 % des transactions), puis on l’applique au test.

In [ ]:
score_train = ...                # -iso.score_samples sur X_train

resultats = []
for part_alertes in [0.005, 0.01, 0.02]:
    seuil = ...                  # quantile (1 - part_alertes) des scores d'entraînement
    alerte = ...                 # score_test > seuil
    p, r = evaluer(y_test, alerte, afficher=False)
    resultats.append({'part visée': part_alertes, 'alertes en test': int(alerte.sum()),
                      'taux d\'alertes': round(alerte.mean(), 4), 'précision': round(p, 3), 'rappel': round(r, 3),
                      'précision max possible': round(min(1, y_test.sum() / max(alerte.sum(), 1)), 3)})
pd.DataFrame(resultats)

## Étape 7 — Effet des hyperparamètres

Comparez plusieurs nombres d’arbres, puis mesurez la stabilité du résultat selon la graine aléatoire.

In [ ]:
for n_arbres in [10, 50, 100, 300]:
    debut = time.time()
    modele = ...                 # IsolationForest(n_estimators=n_arbres, max_samples=256, random_state=42, n_jobs=-1).fit(X_train)
    s = ...                      # -modele.score_samples(X_test)
    duree = time.time() - debut
    print(f'n_estimators = {n_arbres:4d}  ROC-AUC = {roc_auc_score(y_test, s):.3f}  '
          f'AP = {average_precision_score(y_test, s):.3f}  durée = {duree:.1f} s')

In [ ]:
# Stabilité selon la graine aléatoire (configuration par défaut)
aucs, aps = [], []
for graine in range(5):
    modele = ...
    s = ...
    aucs.append(roc_auc_score(y_test, s))
    aps.append(average_precision_score(y_test, s))
print(f'ROC-AUC : moyenne = {np.mean(aucs):.3f}  écart-type = {np.std(aucs):.3f}')
print(f'AP      : moyenne = {np.mean(aps):.3f}  écart-type = {np.std(aps):.3f}')

## Étape 8 — Effet du choix des variables

La fonction `score_ia` (fournie) entraîne un Isolation Forest sur une liste de colonnes et renvoie le ROC-AUC et la précision moyenne sur le test.

In [ ]:
def score_ia(colonnes, graine=42):
    modele = IsolationForest(n_estimators=100, max_samples=256, random_state=graine, n_jobs=-1).fit(train[colonnes])
    s = -modele.score_samples(test[colonnes])
    return roc_auc_score(y_test, s), average_precision_score(y_test, s)

In [ ]:
cols_V = [c for c in df.columns if c.startswith('V')]

print('V1..V28 seulement        :', score_ia(cols_V))
print('V1..V28 + Amount         :', score_ia(cols_V + ['Amount']))

# Ajout de 20 colonnes de bruit sans rapport avec la fraude
rng = np.random.default_rng(0)
for i in range(20):
    train[f'bruit{i}'] = rng.normal(size=len(train))
    test[f'bruit{i}'] = rng.normal(size=len(test))
cols_bruit = [f'bruit{i}' for i in range(20)]
print('V1..V28 + 20 colonnes de bruit :', ...)

## Étape 9 — Inspecter les alertes

In [ ]:
resultat = test.assign(score=score_test)
resultat['rang'] = resultat['score'].rank(ascending=False)

# 10 transactions les plus suspectes (colonnes Amount, Class, score)
...

# Rang (1 = la plus suspecte) des fraudes du test : médiane, et rang de la fraude la moins bien classée
rangs_fraudes = ...
print('Rang médian des fraudes :', ...)
print('Rang de la fraude la moins bien classée :', ...)

## Questions

Répondez dans les cellules de texte situées sous chaque question.

**Question 1.** Quel est le taux de fraude du jeu de données ? Que vaudrait l’accuracy d’un modèle qui répond toujours « pas de fraude » ?

*Votre réponse :*

**Question 2.** Pourquoi découpe-t-on dans le temps ? Pourquoi n’a-t-on pas standardisé les variables ? Pourquoi la colonne `Class` ne doit-elle pas faire partie des variables d’entrée ?

*Votre réponse :*

**Question 3.** Avec `contamination='auto'`, combien d’alertes le modèle lève-t-il dans le test, par rapport au nombre de fraudes ? Ce réglage est-il utilisable en production ?

*Votre réponse :*

**Question 4.** Interprétez le ROC-AUC et la précision moyenne. Pourquoi la précision moyenne doit-elle être comparée au taux de fraude ? Combien de transactions faut-il examiner pour retrouver la moitié des fraudes du test ?

*Votre réponse :*

**Question 5.** Pour une équipe d’analystes qui peut examiner 1 % des transactions, quel seuil retenez-vous ? Quelle est la précision maximale théorique et pourquoi la précision observée reste-t-elle faible ?

*Votre réponse :*

**Question 6.** Que montrent l’étude du nombre d’arbres, des graines et de l’ajout de colonnes de bruit ? Citez deux limites de l’Isolation Forest observées dans ce TP.

*Votre réponse :*

## Pour aller plus loin (facultatif)

1. Ajoutez une variable `heure = (Time // 3600) % 24` et mesurez son effet.
2. Entraînez l’Isolation Forest sur `Amount` seul, puis sur quelques variables `V` seulement : quelles variables apportent le plus ?
3. Comparez avec un modèle supervisé simple (régression logistique avec `class_weight='balanced'`) entraîné sur les mêmes données d’entraînement avec les étiquettes. Que gagne-t-on en utilisant les étiquettes ? (Préfiguration des séances 7 à 9.)